In [2]:


facts = {}
# Fired Rules
# Stores the names of fired rules.
fired_rules = []
# User Interface

# Collects facts from the user.
def ask_yes_no(question):
    """
    Read a yes/no answer from the user.
    قراءة إجابة نعم أو لا من المستخدم.
    """

    while True:

        answer = input(question + " (yes/no): ").strip().lower()

        if answer in ("yes", "y"):
            return True

        elif answer in ("no", "n"):
            return False

        else:
            print("Invalid input! Please enter yes or no.")
# Collect Facts

print("=" * 90)
print("        Coffee Addiction Expert System")
print("=" * 90)

questions = {
    "daily": "1. Do you drink coffee every day?",
    "more_than_3": "2. Do you drink more than 3 cups per day?",
    "headache": "3. Do you get headaches if you skip coffee?",
    "sleepy": "4. Do you feel sleepy without coffee?",
    "morning": "5. Do you drink coffee immediately after waking up?",
    "night": "6. Do you drink coffee late at night?",
    "irritable": "7. Do you become irritable without coffee?",
    "reduce": "8. Do you find it difficult to reduce coffee consumption?",
    "focus": "9. Do you need coffee to concentrate?",
    "not_tired": "10. Do you drink coffee even when you are not tired?"
}

for fact, question in questions.items():
    facts[fact] = ask_yes_no(question)

# Display Collected Facts

print("\nCollected Facts")
print("-" * 40)

for fact, value in facts.items():
    print(f"{fact:20}: {value}")

print()

# Rule Base (Knowledge Base)

# Each rule contains:
# IF       -> Conditions
# OP       -> AND / OR
# THEN     -> New inferred fact
# priority -> Used for conflict resolution


rules = {

    # Level 1 Rules (Direct inferences from user input)
    # قواعد المستوى الأول (استنتاجات مباشرة من إدخال المستخدم)

    "R1": {
        "IF": ["daily", "more_than_3"],
        "OP": "AND",
        "THEN": "high_intake",
        "priority": 1
    },

    "R2": {
        "IF": ["headache", "sleepy"],
        "OP": "AND",
        "THEN": "withdrawal",
        "priority": 1
    },

    "R3": {
        "IF": ["morning"],
        "OP": "AND", # Simple fact, AND with single condition means IF condition is true
        "THEN": "morning_need",
        "priority": 1
    },

    "R4": {
        "IF": ["night"],
        "OP": "AND",
        "THEN": "late_use",
        "priority": 1
    },

    "R5": {
        "IF": ["irritable"],
        "OP": "AND",
        "THEN": "mood_change",
        "priority": 1
    },

    "R6": {
        "IF": ["reduce"],
        "OP": "AND",
        "THEN": "cannot_reduce",
        "priority": 1
    },

    "R7": {
        "IF": ["focus"],
        "OP": "AND",
        "THEN": "focus_dependency",
        "priority": 1
    },

    "R8": {
        "IF": ["not_tired"],
        "OP": "AND",
        "THEN": "habit_drinking",
        "priority": 1
    },

    "R14": { # Example of OR Rule
        "IF": ["headache", "irritable"],
        "OP": "OR",
        "THEN": "possible_withdrawal",
        "priority": 1 # Lower priority than R2 which is more specific withdrawal
    },

    "R21": {
        "IF": ["daily"],
        "OP": "AND",
        "THEN": "coffee_user",
        "priority": 1
    },


    # Level 2 Rules (Chained inferences from Level 1 facts)
    # قواعد المستوى الثاني (استنتاجات متسلسلة من حقائق المستوى الأول)


    "R9": { # Chained rule: R1 + R2 -> R9
        "IF": ["high_intake", "withdrawal"],
        "OP": "AND",
        "THEN": "strong_dependency",
        "priority": 2
    },

    "R10": { # Chained rule: R3 + R7 -> R10
        "IF": ["morning_need", "focus_dependency"],
        "OP": "AND",
        "THEN": "mental_dependency",
        "priority": 2
    },

    "R11": { # Chained rule: R6 + R8 -> R11
        "IF": ["cannot_reduce", "habit_drinking"],
        "OP": "AND",
        "THEN": "behavioral_dependency",
        "priority": 2
    },

    "R12": { # Chained rule: R4 + R1 -> R12
        "IF": ["late_use", "high_intake"],
        "OP": "AND",
        "THEN": "sleep_problem",
        "priority": 2
    },

    "R13": { # Chained rule: R2 + R5 -> R13
        "IF": ["withdrawal", "mood_change"],
        "OP": "AND",
        "THEN": "physical_dependency",
        "priority": 2
    },


    # Level 3 Rules (Candidates for final conclusions)

    "R15": { # 3-step rule chain example: R1+R2->R9, R6+R8->R11, then R9+R11->R15
        "IF": ["strong_dependency", "behavioral_dependency"],
        "OP": "AND",
        "THEN": "severe_candidate",
        "priority": 3
    },

    "R16": {
        "IF": ["mental_dependency", "physical_dependency"],
        "OP": "AND",
        "THEN": "moderate_candidate",
        "priority": 3
    },

    "R17": {
        "IF": ["behavioral_dependency"],
        "OP": "AND",
        "THEN": "mild_candidate",
        "priority": 3
    },


    # Final Conclusion Rules (Highest priority for conflict resolution)

    "R18": {
        "IF": ["severe_candidate"],
        "OP": "AND",
        "THEN": "severe_addiction",
        "priority": 5 # Highest priority conclusion
    },

    "R19": {
        "IF": ["moderate_candidate"],
        "OP": "AND",
        "THEN": "moderate_addiction",
        "priority": 4
    },

    "R20": {
        "IF": ["mild_candidate"],
        "OP": "AND",
        "THEN": "mild_dependence",
        "priority": 3
    }
}

# Inference Engine (Forward Chaining)
# Uses Forward Chaining until no new facts are added.
# Rules are processed based on priority to resolve conflicts.
changed = True

while changed: # Loop continues as long as new facts are inferred
    changed = False

    # Sort rules by priority (higher priority rules are evaluated first)
    sorted_rules = sorted(
        rules.items(),
        key=lambda item: item[1]["priority"],
        reverse=True
    )

    for rule_name, rule in sorted_rules:

        # If the 'THEN' fact is already known, skip this rule
        if facts.get(rule["THEN"], False):
            continue
        satisfied = False
        # Check if the rule's conditions are met

        if rule["OP"] == "AND":
            # All conditions must be true for AND operation

            satisfied = all(
                facts.get(condition, False)
                for condition in rule["IF"]
            )
        else: # OP == "OR"
            # At least one condition must be true for OR operation

            satisfied = any(
                facts.get(condition, False)
                for condition in rule["IF"]
            )

        if satisfied: # If rule is satisfied, infer new fact
            facts[rule["THEN"]] = True
            fired_rules.append(rule_name)
            changed = True # A new fact was added, so continue the loop

# Results Display
# Displays fired rules, inferred facts,
# final conclusion and recommendation.

print("\n" + "=" * 60)
print("Rules Fired")
print("=" * 60)

if fired_rules:
    # Display fired rules, sorted for clarity

    for rule in sorted(set(fired_rules)): # Using set to show unique rules
        print(rule)
else:
    print("No rules were fired.")

# Final Facts

print("\n" + "=" * 60)
print("Final Facts")
print("=" * 60)

# Display all facts that are true
for fact, value in facts.items():
    if value:
        print("-", fact)

# Final Diagnosis (Conflict Resolution)

# If multiple conclusions are inferred, the highest-priority
# conclusion is selected based on the order of checks.

conclusion = "Not Applicable"
recommendation = "No specific recommendation found."

# Check for conclusions in order of severity (highest priority first)
if facts.get("severe_addiction"):

    conclusion = "Severe Coffee Addiction"

    recommendation = (
        "1. Gradually reduce your coffee intake.\n"
        "2. Consult a healthcare professional.\n"
        "3. Avoid drinking coffee late at night.\n"
        "4. Replace coffee with decaffeinated drinks and seek support."
    )

elif facts.get("moderate_addiction"):

    conclusion = "Moderate Coffee Dependence"

    recommendation = (
        "1. Reduce your coffee consumption gradually (e.g., one cup less per week).\n"
        "2. Drink more water and herbal teas during the day.\n"
        "3. Avoid coffee before bedtime and limit afternoon intake.\n"
        "4. Pay attention to withdrawal symptoms and adjust slowly."
    )

elif facts.get("mild_dependence"):

    conclusion = "Mild Coffee Dependence"

    recommendation = (
        "1. Monitor your coffee intake and be mindful of patterns.\n"
        "2. Try caffeine-free alternatives when possible.\n"
        "3. Avoid increasing the number of cups or strength.\n"
        "4. Ensure good sleep and hydration to reduce reliance on coffee."
    )

elif facts.get("coffee_user"):
    # If no addiction or dependence, but still a regular coffee user

    conclusion = "Regular Coffee User (Not Addicted)"
    recommendation = (
        "1. Your coffee consumption appears healthy and controlled.\n"
        "2. Continue drinking coffee in moderation.\n"
        "3. Maintain a balanced lifestyle with adequate rest and hydration."
    )

else:
    conclusion = "Not a Regular Coffee User"
    recommendation = (
        "1. You do not appear to be a regular coffee user.\n"
        "2. No signs of coffee dependence were detected.\n"
        "3. Maintain your current habits if they contribute to your well-being."
    )


# Display Final Result

print("\n" + "=" * 60)
print("Final Conclusion")
print("=" * 60)

print(conclusion)

print("\n" + "=" * 60)
print("Recommendation")
print("=" * 60)

print(recommendation)

print("\nThank you for using the Coffee Addiction Expert System.")

        Coffee Addiction Expert System
1. Do you drink coffee every day? (yes/no): n
2. Do you drink more than 3 cups per day? (yes/no): y
3. Do you get headaches if you skip coffee? (yes/no): y
4. Do you feel sleepy without coffee? (yes/no): y
5. Do you drink coffee immediately after waking up? (yes/no): 
Invalid input! Please enter yes or no.
5. Do you drink coffee immediately after waking up? (yes/no): y
6. Do you drink coffee late at night? (yes/no): n
7. Do you become irritable without coffee? (yes/no): y
8. Do you find it difficult to reduce coffee consumption? (yes/no): n
9. Do you need coffee to concentrate? (yes/no): y
10. Do you drink coffee even when you are not tired? (yes/no): n

Collected Facts
----------------------------------------
daily               : False
more_than_3         : True
headache            : True
sleepy              : True
morning             : True
night               : False
irritable           : True
reduce              : False
focus               : 